# Customize Hyperparameter Optimization
In this notebook, we will look at how to customize hyperparameter optimization for a model:
- Use different hyperparameter selection, or settings for the standard models
- How to define a custom model, and use it in the hyperparameter optimization
- How to use the pipeline-utility classes `PipelineWithHyperparameterRooting` and `ColumnTransformerWithHyperparameterRooting` 

### Example
We will look at the example regression problem, and generate a catboost regressor model, and a cross-validation class

In [1]:
from pathlib import Path

import pandas as pd
from sklearn import pipeline as sklearn_pipeline
from sklearn.model_selection import GroupKFold
import sklearn.metrics as skl_metrics
import mother.ml as ml
import mother.optimization as opt
from mother import cv as cv_module
from mother import feature_generation as fg
from mother.preprocessing import SmilesToMolTransformer, StandardizerTransformer

In [2]:
input_file: Path = Path("../freesolv_train.csv")
data: pd.DataFrame = pd.read_csv(input_file, sep=",")
data = data.head(100)  # limit data to 100 rows for testing
data = data[["iupac", "smiles", "expt"]]

In [3]:
# preprocessing
preprocessor: sklearn_pipeline.Pipeline = sklearn_pipeline.Pipeline(
    [
        (
            "smiles_standardizer",
            StandardizerTransformer(flags=["STANDARDIZE", "NEUTRALIZE", "DESALT"]),  # standardize, normalize, desalt
        ),
        ("smiles_to_mol", SmilesToMolTransformer()),
        # Add other column transformations here if needed
    ],
    memory=None,
)
preprocessor.set_output(transform="pandas")

structure_data: pd.Series = data["smiles"]
mol_data: pd.DataFrame = preprocessor.fit_transform(structure_data)

[06:15:55] Initializing Normalizer
[06:15:55] Initializing Normalizer
[06:15:55] Initializing MetalDisconnector
[06:15:55] Initializing Normalizer


In [4]:
# feature generation
feature_generator = sklearn_pipeline.FeatureUnion(
    transformer_list=[
        ("maccs", fg.MaccsFingerprints()),
        ("morgan", fg.MorganFingerprints()),
        ("desc", fg.ChemicalDescriptors()),
    ],
).set_output(transform="pandas")
features: pd.DataFrame = feature_generator.fit_transform(mol_data["Molecule"])

In [ ]:
# cv grouping
cv = GroupKFold(n_splits=5)

groups_engine = cv_module.TanimotoGroupingFromMols(
    similarity_threshold=0.3, radius=2, fp_size=2048, include_chirality=True
)

groups: pd.DataFrame = groups_engine.set_output(transform="pandas").fit_transform(mol_data["Molecule"])
groups.head()

,tanimoto_cluster
0,0.0
1,1.0
2,2.0
3,3.0
4,4.0


### Use the standard Catboost model with different hyperparameters
Instead of using the `model.get_hyperparameter_space()` method, we can directly define the hyperparameters for the model in another function. We need an `optuna.trial.Trial` object for this.

In [6]:
model = ml.CatboostRegressorMother(target_type="single_target", logging_level="Silent", max_depth=2, n_estimators=100)

In [7]:
from optuna.trial import Trial

In [8]:
def custom_hyperparameter_space_function(X, y, trial: Trial) -> dict:
    """only adjust the learning rate of the catboost model"""
    return {
        "learning_rate": trial.suggest_float("learning_rate", 0.000001, 0.5, log=True),
    }

In [9]:
# do the hyperparameter optimization
# - Default Parameter are also optional

tuner = opt.MotherTuner(
    scorer=skl_metrics.get_scorer("r2"),
    n_trials_optuna=1,  # number of trials for hyperparameter optimization
    n_threads_optuna=10,  # parallel threads for cross-validation evaluation
    n_startup_trials=1,  # number of random trials before using optuna
    tuning_direction="maximize",  # maximize or minimize the scorer
)

model_tuned = tuner.optimize(
    model,
    features,
    data["expt"],
    cv,
    hyperparameter_space_function=custom_hyperparameter_space_function,
    groups=groups.values,
)

/workspaces/mother-ml/.venv/lib/python3.11/site-packages/optuna/_experimental.py:31: ExperimentalWarning: Argument ``multivariate`` is an experimental feature. The interface can change in the future.
  warnings.warn(
/workspaces/mother-ml/.venv/lib/python3.11/site-packages/optuna/_experimental.py:31: ExperimentalWarning: Argument ``group`` is an experimental feature. The interface can change in the future.
  warnings.warn(
/workspaces/mother-ml/.venv/lib/python3.11/site-packages/optuna/_experimental.py:31: ExperimentalWarning: Argument ``constant_liar`` is an experimental feature. The interface can change in the future.
  warnings.warn(


### Custom Model With Hyperparameters
Custom Models can be used out-of-the-box, if they are defined as a sklearn `estimator`. The additional methods `get_hyperparameter_space` and`default_parameters` are useful for hyperparameter optimization. 

The pipeline classes `PipelineWithHyperparameterRooting` and
`ColumnTransformerWithHyperparameterRooting` will search for these methods, and take care of hyperparameter handling in stacked / combined processing steps.

In this example, we will first build a custom model (we use sklearn Lasso), and combine it with preprocessing steps. 

In [10]:
from sklearn.linear_model import Lasso
from sklearn.decomposition import PCA
from mother import ml

In [11]:
class CustomLassoModel(Lasso):
    def get_hyperparameter_space(self, X, y, trial: Trial, prefix: str = "") -> dict:
        return {
            prefix + "alpha": trial.suggest_float(prefix + "alpha", 1e-6, 1e1, log=True),
        }


model = CustomLassoModel()

The `prefix` argument is used to separate the hyperparameters of different steps of pipelines / column transformers.

We can now create a pipeline with a preprocessing step. Let us do a PCA first, and then use the Lasso model.

In [12]:
class CustomPCA(PCA):
    """overload PCA to have a hyperparameter space"""

    def get_hyperparameter_space(self, X, y, trial: Trial, prefix: str = "") -> dict:
        return {prefix + "n_components": trial.suggest_int(prefix + "n_components", 1, 10)}

    def default_parameters(self, prefix: str = "") -> dict:
        return {prefix + "n_components": 5}

In [13]:
model_with_pca = ml.PipelineWithHyperparameterRooting([("PCA", CustomPCA()), ("model", model)])

In [14]:
tuner = opt.MotherTuner(
    scorer=skl_metrics.get_scorer("r2"),
    n_trials_optuna=1,  # number of trials for hyperparameter optimization
    n_threads_optuna=10,  # parallel threads for cross-validation evaluation
    n_startup_trials=1,  # number of random trials before using optuna
    tuning_direction="maximize",  # maximize or minimize the scorer
)

# tune the LASSO model alone
model_tuned = tuner.optimize(
    model,
    features,
    data[["expt"]],
    cv,
    hyperparameter_space_function=model.get_hyperparameter_space,
    groups=groups.values,
)

/workspaces/mother-ml/.venv/lib/python3.11/site-packages/optuna/_experimental.py:31: ExperimentalWarning: Argument ``multivariate`` is an experimental feature. The interface can change in the future.
  warnings.warn(
/workspaces/mother-ml/.venv/lib/python3.11/site-packages/optuna/_experimental.py:31: ExperimentalWarning: Argument ``group`` is an experimental feature. The interface can change in the future.
  warnings.warn(
/workspaces/mother-ml/.venv/lib/python3.11/site-packages/optuna/_experimental.py:31: ExperimentalWarning: Argument ``constant_liar`` is an experimental feature. The interface can change in the future.
  warnings.warn(
/workspaces/mother-ml/.venv/lib/python3.11/site-packages/sklearn/linear_model/_coordinate_descent.py:697: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 3.367e-01, tolerance: 6.993e-02
  model = cd_fast.enet_coordinat

In [15]:
model_tuned

CustomLassoModel(alpha=0.0004185822729546974)

In [16]:
# tune the PCA/LASSO pipeline
model_with_pca_tuned = tuner.optimize(
    model_with_pca,
    features,
    data["expt"],
    cv,
    hyperparameter_space_function=model_with_pca.get_hyperparameter_space,
    groups=groups.values,
)

model_with_pca_tuned

PipelineWithHyperparameterRooting(steps=[('PCA', CustomPCA(n_components=5)),
                                         ('model',
                                          CustomLassoModel(alpha=4.518560951024111))])

In [17]:
model_with_pca_tuned["PCA"]

CustomPCA(n_components=5)

In [18]:
model_with_pca_tuned["model"]

CustomLassoModel(alpha=4.518560951024111)